# Travel Agent with Redis Cloud, Snowflake Cortex, Free Web Search & Weather API

- Snowflake Cortex for LLM (chat) & embeddings (arctic‑embed‑l‑v2.0)
- Redis Cloud for long‑term vector memory + checkpoints + transcripts
- DuckDuckGo web search (free, no key) via `ddgs`
- OpenWeatherMap API for accurate weather (free tier, 1,000 calls/day)
- Date utility (no external call)
- Tool‑call cap (3 per turn) and robust JSON parsing
- Conversation selector on startup
- In‑chat commands: `exit`, `debug`, `history`, `memories`

# Cell 1 – Install dependencies

In [1]:
# ddgs is the renamed successor to duckduckgo-search
%pip install langgraph langgraph-checkpoint redis redisvl ulid pydantic requests python-dotenv ddgs beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


# Cell 2 – Environment variables

In [2]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}: ")

_set_env("SNOWFLAKE_ACCOUNT")
_set_env("SNOWFLAKE_PAT")
_set_env("MODEL")
_set_env("REDIS_URL")
_set_env("OPENWEATHER_API_KEY")   # get free key from openweathermap.org

# Cell 3 – Connect to Redis

In [3]:
from redis import Redis
REDIS_URL = os.environ["REDIS_URL"]
redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()
print("✅ Connected to Redis")

✅ Connected to Redis


# Cell 4 – Pydantic models

In [4]:
import ulid
from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

class MemoryType(str, Enum):
    EPISODIC = "episodic"
    SEMANTIC = "semantic"

class Memory(BaseModel):
    content: str
    memory_type: MemoryType
    metadata: str

class StoredMemory(Memory):
    id: str
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None

print("✅ Models ready")

✅ Models ready


# Cell 5 – RedisVL vector index

In [5]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema

VECTOR_DIM = 1024

memory_schema = IndexSchema.from_dict({
    "index": {"name": "agent_memories", "prefix": "memory", "key_separator": ":", "storage_type": "json"},
    "fields": [
        {"name": "content", "type": "text"},
        {"name": "memory_type", "type": "tag"},
        {"name": "metadata", "type": "text"},
        {"name": "created_at", "type": "text"},
        {"name": "user_id", "type": "tag"},
        {"name": "memory_id", "type": "tag"},
        {"name": "thread_id", "type": "tag"},
        {"name": "embedding", "type": "vector", "attrs": {"algorithm": "flat", "dims": VECTOR_DIM, "distance_metric": "cosine", "datatype": "float32"}},
    ],
})

long_term_memory_index = SearchIndex(schema=memory_schema, redis_client=redis_client, validate_on_load=True)
long_term_memory_index.create(overwrite=True)
print("✅ Long‑term memory index ready")

✅ Long‑term memory index ready


# Cell 6 – Snowflake embedding helper

In [16]:
import requests, json
ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
PAT = os.environ["SNOWFLAKE_PAT"]
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"

def embed_text(text: str) -> List[float]:
    # Safety: if text is a list, convert to string
    if isinstance(text, list):
        text = " ".join(str(item) for item in text if item) if text else ""
    if not text or not isinstance(text, str):
        raise ValueError("embed_text requires a non‑empty string")
    url = f"https://{ACCOUNT}.snowflakecomputing.com/api/v2/cortex/inference:embed"
    headers = {"Authorization": f"Bearer {PAT}", "Content-Type": "application/json"}
    payload = {"model": EMBED_MODEL, "input": text}
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Embed API error {response.status_code}: {response.text}")
    return response.json()["data"][0]["embedding"]

# Cell 7 – Memory operations (store, retrieve, deduplicate)

In [17]:
import logging
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")
SYSTEM_USER_ID = "system"

def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    embedding = embed_text(content)
    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type.value)
    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)
    q = VectorRangeQuery(
        vector=embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    return len(long_term_memory_index.query(q)) > 0

def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
) -> None:
    if metadata is None:
        metadata = "{}"
    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory exists — skipping.")
        return
    embedding = embed_text(content)
    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id or "",
    }
    long_term_memory_index.load([memory_data])
    logger.info(f"💾 Stored [{memory_type.value}] memory: {content!r}")

def retrieve_memories(
    query: str,
    memory_type: Optional[MemoryType] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.4,
    limit: int = 5,
) -> List[StoredMemory]:
    if not query or not query.strip():
        return []   # nothing to retrieve
    embedding = embed_text(query)
    filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]
    if memory_type:
        filters.append(f"@memory_type:{{{memory_type.value}}}")
    if thread_id:
        filters.append(f"@thread_id:{{{thread_id}}}")
    q = VectorRangeQuery(
        vector=embedding,
        return_fields=["content", "memory_type", "metadata", "created_at", "memory_id", "thread_id", "user_id"],
        num_results=limit,
        vector_field_name="embedding",
        distance_threshold=distance_threshold,
        dialect=2,
    )
    q.set_filter(" ".join(filters))
    results = long_term_memory_index.query(q)
    memories = []
    for doc in results:
        try:
            memories.append(StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id") or None,
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            ))
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
    return memories

print("✅ Memory operations ready")

✅ Memory operations ready


# Cell 8 – Tools (memory, web search, weather, date)

> **Fix**: switched from deprecated `duckduckgo-search` (DDGS) to `ddgs` package.  
> The web_search_tool now receives the **full** query string, not just a regex capture group.

In [18]:
from ddgs import DDGS
import re, requests, datetime

# ── Memory tools ───────────────────────────────────────────────────────────
def store_memory_tool(
    content: str,
    memory_type: str,
    metadata: Optional[dict] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,   # defaults to None → user‑scoped
) -> str:
    try:
        mem_type = MemoryType(memory_type)
        store_memory(content, mem_type, user_id, thread_id, str(metadata) if metadata else None)
        return f"✅ Stored [{mem_type.value}] memory: {content}"
    except Exception as e:
        return f"❌ Error storing memory: {e}"

def retrieve_memories_tool(
    query: str,
    memory_type: Optional[str] = None,
    limit: int = 5,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type) if memory_type else None
        memories = retrieve_memories(query, mem_type, user_id, thread_id, limit=limit)
        if not memories:
            return "No relevant memories found."
        lines = ["🧠 Long-term memories:"]
        for m in memories:
            lines.append(f"  - [{m.memory_type.value}] {m.content}")
        return "\n".join(lines)
    except Exception as e:
        return f"❌ Error retrieving memories: {e}"

# ── Date tool (no external call) ──────────────────────────────────────────
def get_current_date_tool(**kwargs) -> str:
    """Return today's date and tomorrow's date."""
    now = datetime.datetime.now()
    return (
        f"Today is {now.strftime('%A, %B %d, %Y')}. "
        f"Tomorrow will be {(now + datetime.timedelta(days=1)).strftime('%A, %B %d, %Y')}."
    )

# ── Weather tool (OpenWeatherMap) ─────────────────────────────────────────
def get_weather_tool(city: str, country_code: str = "", units: str = "metric", **kwargs) -> str:
    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return "ERROR: OPENWEATHER_API_KEY not set."
    q = f"{city},{country_code}" if country_code else city
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": q, "appid": api_key, "units": units}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        d = r.json()
        temp  = d["main"]["temp"]
        feels = d["main"]["feels_like"]
        cond  = d["weather"][0]["description"].capitalize()
        hum   = d["main"]["humidity"]
        wind  = d["wind"]["speed"]
        unit_sym = "C" if units == "metric" else "F"
        return (
            f"Weather in {d['name']}, {d['sys']['country']}:\n"
            f"🌡️  Temp: {temp}°{unit_sym}  (feels like {feels}°{unit_sym})\n"
            f"☁️  {cond}\n"
            f"💧 Humidity: {hum}%  💨 Wind: {wind} m/s"
        )
    except Exception as e:
        return f"Error fetching weather: {e}"

# ── Web search tool (ddgs) ─────────────────────────────────────────────────
def web_search_tool(query: str, max_results: int = 5, **kwargs) -> str:
    """
    Search the web using DuckDuckGo.
    IMPORTANT: pass the FULL query – do not strip leading keywords.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return f"No web results found for '{query}'."
        output = [f"🌐 Web search results for '{query}':"]
        for r in results:
            title   = r.get("title", "No title")
            snippet = re.sub(r"\s+", " ", r.get("body", "")).strip()
            url     = r.get("href", "")
            output.append(f"- **{title}**\n  {snippet}\n  🔗 {url}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error performing web search: {e}"

# ── Tool registry ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "store_memory",
        "func": store_memory_tool,
        "description": (
            "Store a fact with memory_type='episodic' or 'semantic'. "
            "Example: {\"tool\": \"store_memory\", \"arguments\": {\"content\": \"Rahmath likes spicy food\", \"memory_type\": \"episodic\"}}"
        ),
    },
    {
        "name": "retrieve_memories",
        "func": retrieve_memories_tool,
        "description": (
            "Recall past facts. "
            "Example: {\"tool\": \"retrieve_memories\", \"arguments\": {\"query\": \"pin code\", \"memory_type\": \"episodic\", \"limit\": 3}}"
        ),
    },
    {
        "name": "get_current_date",
        "func": get_current_date_tool,
        "description": "Get today's and tomorrow's date. No arguments needed.",
    },
    {
        "name": "get_weather",
        "func": get_weather_tool,
        "description": "Get current weather. Parameters: city (str), country_code (optional, e.g. 'IN'), units ('metric' or 'imperial').",
    },
    {
        "name": "web_search",
        "func": web_search_tool,
        "description": "Search the web. Parameters: query (str – the FULL search query), max_results (int, default=5).",
    },
]

print("✅ Tools ready")

✅ Tools ready


# Cell 9 – Snowflake Cortex LLM (improved JSON extraction)

In [19]:
import re, json, ulid, requests, logging
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks import CallbackManagerForLLMRun
from typing import Any, List, Optional, Sequence

logger = logging.getLogger(__name__)

def extract_tool_call(text: str) -> Optional[dict]:
    """Extract a JSON object with 'tool' and 'arguments' keys, even from malformed text."""
    # Remove markdown code fences
    cleaned = re.sub(r"```(?:json)?\n?|```", "", text).strip()
    start = cleaned.find("{")
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : i + 1]
                try:
                    obj = json.loads(candidate)
                    if "tool" in obj and "arguments" in obj:
                        return obj
                except json.JSONDecodeError:
                    pass
                break
    return None


class SnowflakeCortexLLM(BaseChatModel):
    account: str
    pat: str
    model: str
    temperature: float = 0.0

    def _generate(self, messages: List[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        api_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                api_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                api_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                api_messages.append({"role": "assistant", "content": msg.content})
            elif isinstance(msg, ToolMessage):
                api_messages.append({"role": "user", "content": f"[Tool result for '{msg.name}']:\n{msg.content}"})

        url = f"https://{self.account}.snowflakecomputing.com/api/v2/cortex/v1/chat/completions"
        headers = {"Authorization": f"Bearer {self.pat}", "Content-Type": "application/json"}
        payload = {"model": self.model, "messages": api_messages, "temperature": self.temperature}

        response = requests.post(url, headers=headers, json=payload)
        if response.status_code != 200:
            raise RuntimeError(f"Snowflake API error {response.status_code}: {response.text}")

        data = response.json()
        raw_content = data["choices"][0]["message"]["content"].strip()
        logger.debug(f"Raw LLM output: {raw_content!r}")

        # Detect and discard completely malformed / empty responses
        if raw_content in ("}", "", "{}"):
            raw_content = "I'm sorry, I couldn't process that. Could you rephrase?"

        tool_call = extract_tool_call(raw_content)
        clean_content = raw_content
        if tool_call:
            clean_content = re.sub(r"\{[\"']tool[\"'][\s\S]*?\}", "", raw_content).strip()
            if not clean_content:
                clean_content = "Using a tool to help you."

        ai_msg = AIMessage(
            content=clean_content,
            tool_calls=[
                {
                    "name": tool_call["tool"],
                    "args": tool_call["arguments"],
                    "id": f"call_{ulid.ULID()}",
                }
            ] if tool_call else [],
        )
        return ChatResult(generations=[ChatGeneration(message=ai_msg)])

    @property
    def _llm_type(self) -> str:
        return "snowflake-cortex"


account = os.environ["SNOWFLAKE_ACCOUNT"]
pat     = os.environ["SNOWFLAKE_PAT"]
model   = os.environ["MODEL"]
llm     = SnowflakeCortexLLM(account=account, pat=pat, model=model, temperature=0.0)
print("✅ Snowflake Cortex LLM ready")

✅ Snowflake Cortex LLM ready


# Cell 10 – Conversation persistence (transcripts + selector)

In [20]:
from datetime import datetime
import json, logging

CONV_PREFIX = "conversation"

def save_transcript(state, thread_id: str, user_id: str) -> None:
    transcript = []
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            transcript.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                transcript.append({"role": "assistant", "content": content})
        elif isinstance(m, ToolMessage):
            transcript.append({"role": "tool", "name": m.name, "content": m.content})
        elif isinstance(m, SystemMessage):
            transcript.append({"role": "summary", "content": m.content})
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    redis_client.set(key, json.dumps(transcript))
    logger.info(f"💾 Transcript saved → {key} ({len(transcript)} turns)")

def load_transcript(thread_id: str, user_id: str) -> list:
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    raw = redis_client.get(key)
    return json.loads(raw) if raw else []

def get_all_transcript_keys(user_id: str) -> list:
    pattern = f"{CONV_PREFIX}:{user_id}:*"
    return sorted([k.decode() for k in redis_client.keys(pattern)])

def select_conversation(user_id: str = "demo_user") -> tuple:
    """Interactive selector; returns (thread_id, transcript)."""
    keys = get_all_transcript_keys(user_id)
    print("\n" + "=" * 55)
    print("  📚 CONVERSATION SELECTOR")
    print("=" * 55)
    if not keys:
        print("  No previous conversations found. Starting new.\n")
        thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        return thread_id, []
    print("  [0] 🆕 Start a new conversation\n")
    for i, key in enumerate(keys, start=1):
        thread_part = key.split(":", 2)[2] if key.count(":") >= 2 else key
        raw = redis_client.get(key)
        turns = len(json.loads(raw)) if raw else 0
        preview = ""
        if raw:
            for turn in json.loads(raw):
                if turn.get("role") == "user":
                    preview = turn.get("content", "")[:60]
                    break
        print(f"  [{i}] 🗂  {thread_part}")
        print(f"       {turns} turns  |  \"{preview}{'...' if len(preview) == 60 else ''}\"")
        print()
    print("=" * 55)
    while True:
        try:
            choice = input(f"  Choose [0-{len(keys)}]: ").strip()
        except (EOFError, KeyboardInterrupt):
            choice = "0"
        if choice == "0":
            thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            print(f"\n  🆕 New conversation started: {thread_id}\n")
            return thread_id, []
        if choice.isdigit() and 1 <= int(choice) <= len(keys):
            idx = int(choice) - 1
            chosen_key = keys[idx]
            thread_id = chosen_key.split(":", 2)[2]
            transcript = json.loads(redis_client.get(chosen_key))
            print(f"\n  📂 Resuming: {thread_id} ({len(transcript)} turns)\n")
            return thread_id, transcript
        print(f"  ❌ Invalid choice. Enter a number between 0 and {len(keys)}.")

def print_transcript(thread_id: str, user_id: str = "demo_user") -> None:
    transcript = load_transcript(thread_id, user_id)
    if not transcript:
        print(f"No transcript found for user={user_id!r} thread={thread_id!r}")
        return
    print(f"\n{'='*60}")
    print(f"TRANSCRIPT  user={user_id}  thread={thread_id}")
    print(f"{'='*60}")
    for turn in transcript:
        role = turn["role"].upper()
        name = f" ({turn['name']})" if turn.get("name") else ""
        print(f"\n[{role}{name}]\n{turn.get('content', '')}")
    print()

print("✅ Conversation transcript helpers ready")

✅ Conversation transcript helpers ready


# Cell 11 – LangGraph workflow

**Fixes applied in this cell:**
1. **Web-search regex** now captures the *complete* user query — it no longer strips the trigger word, and uses named groups to cleanly extract what to search for.
2. **Date regex** is tightened so it won't fire on messages that only mention "today" as part of a news/search request.
3. **General travel / info questions** (`"tell me about hampi"`, `"what are the caves in hampi"`) now route correctly to `web_search_tool` instead of falling through to a broken LLM fallback.
4. **LLM fallback** is kept for true conversational exchanges that none of the above handles, with a clean error message rather than a garbage response.

In [21]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.redis import RedisSaver
from langchain_core.runnables.config import RunnableConfig
import re
from datetime import datetime, timedelta

class RuntimeState(MessagesState):
    tool_call_count: int

MAX_TOOL_CALLS_PER_TURN = 3

SYSTEM_PROMPT = """You are a travel assistant with tools.

Available tools:
1. store_memory(content, memory_type) – content is the fact, memory_type is 'episodic'.
2. retrieve_memories(query) – recall facts.
3. get_current_date() – no args.
4. get_weather(city, country_code, units) – country_code optional, units 'metric' or 'imperial'.
5. web_search(query) – search the web; pass the FULL query string.

To call a tool, respond with ONLY this JSON (no other text):
{"tool": "tool_name", "arguments": {"arg1": "value"}}

If you don't need a tool, reply in plain text.
"""

# ─────────────────────────────────────────────────────────────────────────
# Helper: does the message look like a pure date question?
# ─────────────────────────────────────────────────────────────────────────
_DATE_ONLY_RE = re.compile(
    r"^(?:what(?:'s|\s+is)?\s+(?:the\s+)?|"
    r"tell\s+me\s+(?:the\s+)?)?(?:today|tomorrow|current\s+date|today\'s\s+date|date\s+today)\s*\?*$",
    re.IGNORECASE,
)

# ─────────────────────────────────────────────────────────────────────────
# Helper: build a normalised search query from user input.
# ─────────────────────────────────────────────────────────────────────────
_SEARCH_TRIGGER_RE = re.compile(
    r"^(?:(?:do\s+(?:a|the)\s+)?(?:web\s*)?search\s+for\s+(?:the\s+)?"
    r"|find(?:s+me)?\s+(?:information\s+(?:about|on)\s+)?"
    r"|look(?:s+it)?\s+up\s+"
    r"|search(?:s+for)?\s+"
    r"|news\s+(?:on|about|for)?\s*"
    r"|tell\s+me\s+about\s+"
    r"|what\s+(?:is|are|do\s+you\s+know\s+about)\s+)",
    re.IGNORECASE,
)

def _build_search_query(user_input: str) -> Optional[str]:
    stripped = user_input.strip()
    m = _SEARCH_TRIGGER_RE.match(stripped)
    if m:
        remainder = stripped[m.end():].strip(" ?.")
        if remainder:
            return remainder
    return None

# ─────────────────────────────────────────────────────────────────────────
# Agent node
# ─────────────────────────────────────────────────────────────────────────
def respond_to_user(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    user_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not user_msgs:
        return state

    user_input   = user_msgs[-1].content.strip()
    configurable = config.get("configurable", {}) if config else {}
    user_id      = configurable.get("user_id", SYSTEM_USER_ID)
    thread_id    = configurable.get("thread_id")   # still for context, not for memory storage

    # ── 1. Store memory: name ────────────────────────────────────────────
    store_name = re.search(r"(?:my name is|i am|call me)\s+(\w+)", user_input, re.IGNORECASE)
    if store_name:
        name = store_name.group(1).capitalize()
        store_memory_tool(
            content=f"User's name is {name}",
            memory_type="episodic",
            user_id=user_id,
            # thread_id omitted → user‑scoped
        )
        state["messages"].append(AIMessage(content=f"✅ Got it! I've saved your name as **{name}**."))
        return state

    # ── 2. Store memory: pin code ────────────────────────────────────────
    pin_match = re.search(
        r"(?:pin\s*code|pincode|postal\s*code)\s*(?:is|of|for)?\s*(?:\w+\s*)?(\d{5,6})",
        user_input,
        re.IGNORECASE,
    )
    if pin_match:
        place_match = re.search(
            r"(?:pin\s*code|pincode|postal\s*code)\s*(?:is|of|for)?\s*(\w+)\s*(?:is\s*)?(\d{5,6})",
            user_input,
            re.IGNORECASE,
        )
        pin  = pin_match.group(1)
        place = place_match.group(1).capitalize() if place_match else "Unknown location"
        fact  = f"{place} pin code is {pin}"
        store_memory_tool(
            content=fact,
            memory_type="episodic",
            user_id=user_id,
            # thread_id omitted → user‑scoped
        )
        state["messages"].append(AIMessage(content=f"✅ Saved: {fact}"))
        return state

    # ── 3. Retrieve memory: name ─────────────────────────────────────────
    # FIXED: now matches "whats my name" (without apostrophe)
    if re.search(r"(?:what(?:'?s|\s+is)?\s+my\s+name|do\s+you\s+know\s+my\s+name|my\s+name\?|^name\??$)", user_input, re.IGNORECASE):
        memories = retrieve_memories(
            query="user name",
            memory_type=MemoryType.EPISODIC,
            user_id=user_id,
            limit=1,
            # thread_id omitted → user‑scoped
        )
        answer = memories[0].content if memories else "I don't know your name yet — please tell me!"
        state["messages"].append(AIMessage(content=answer))
        return state

    # ── 4. Retrieve memory: pin code ──────────────────────────────────────
    pin_query = re.search(
        r"(?:pin\s*code|pincode|postal\s*code)\s*(?:of|for)?\s*(\w+)",
        user_input,
        re.IGNORECASE,
    )
    if pin_query and not re.search(r"\d{5,6}", user_input):
        place = pin_query.group(1).capitalize()
        memories = retrieve_memories(
            query=f"{place} pin code",
            memory_type=MemoryType.EPISODIC,
            user_id=user_id,
            limit=1,
            # thread_id omitted → user‑scoped
        )
        if memories:
            state["messages"].append(AIMessage(content=memories[0].content))
        else:
            state["messages"].append(AIMessage(content=f"I don't have the pin code for {place} stored yet."))
        return state

    # ── 5. Date — only for pure date questions ────────────────────────────
    if _DATE_ONLY_RE.match(user_input.strip()):
        now      = datetime.now()
        tomorrow = now + timedelta(days=1)
        answer   = (
            f"Today is **{now.strftime('%A, %B %d, %Y')}**. "
            f"Tomorrow is **{tomorrow.strftime('%A, %B %d, %Y')}**."
        )
        state["messages"].append(AIMessage(content=answer))
        return state

    # ── 6. Weather ────────────────────────────────────────────────────────
    weather_match = re.search(
        r"weather(?:s+(?:in|of|for|at))?\s+([\w\s]+?)(?:\s*,\s*(\w{2}))?\s*\?*$",
        user_input,
        re.IGNORECASE,
    )
    if weather_match:
        city    = weather_match.group(1).strip()
        country = (weather_match.group(2) or "").strip()
        result  = get_weather_tool(city=city, country_code=country, units="metric")
        state["messages"].append(AIMessage(content=result))
        return state

    # ── 7. Explicit web / info search ─────────────────────────────────────
    search_query = _build_search_query(user_input)
    if search_query:
        if re.search(r"news", user_input, re.IGNORECASE):
            search_query = f"{search_query} news 2026"
        result = web_search_tool(query=search_query, max_results=5)
        state["messages"].append(AIMessage(content=result))
        return state

    # ── 8. LLM fallback for genuine conversation ──────────────────────────
    tool_calls_this_turn = state.get("tool_call_count", 0)
    llm_messages = [SystemMessage(content=SYSTEM_PROMPT)]
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            llm_messages.append(HumanMessage(content=m.content))
        elif isinstance(m, AIMessage):
            if m.content and not extract_tool_call(m.content):
                llm_messages.append(AIMessage(content=m.content))
        elif isinstance(m, ToolMessage):
            llm_messages.append(HumanMessage(content=f"[Tool result for '{m.name}']:\n{m.content}"))

    if tool_calls_this_turn >= MAX_TOOL_CALLS_PER_TURN:
        llm_messages.append(HumanMessage(content="You have used the maximum tool calls. Reply directly in plain text."))

    ai_msg = llm.invoke(llm_messages)

    if not ai_msg.content or ai_msg.content.strip() in ("}", ""):
        logger.warning("Malformed LLM response — falling back to web search on user query.")
        result = web_search_tool(query=user_input, max_results=5)
        state["messages"].append(AIMessage(content=result))
        return state

    state["messages"].append(ai_msg)
    return state


# ── execute_tools node ────────────────────────────────────────────────────
def execute_tools(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    ai_msgs = [m for m in state["messages"] if isinstance(m, AIMessage) and getattr(m, "tool_calls", None)]
    if not ai_msgs:
        return state

    latest       = ai_msgs[-1]
    configurable = config.get("configurable", {}) if config else {}
    user_id      = configurable.get("user_id", SYSTEM_USER_ID)

    for tc in latest.tool_calls:
        tool_name = tc["name"]
        tool_args = dict(tc["args"])
        tool_func = next((t["func"] for t in TOOLS if t["name"] == tool_name), None)

        if not tool_func:
            result = f"Tool '{tool_name}' not found."
        else:
            tool_args.setdefault("user_id", user_id)
            # Do NOT inject thread_id – let the tool use its default None
            try:
                result = tool_func(**tool_args)
                logger.info(f"Tool '{tool_name}' result: {result!r}")
            except Exception as e:
                result = f"Error executing tool '{tool_name}': {e}"
                logger.error(result)

        state["messages"].append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tool_name))

    state["tool_call_count"] = state.get("tool_call_count", 0) + 1
    return state


# ── summarize node ────────────────────────────────────────────────────────
MESSAGE_THRESHOLD = 8

def summarize_conversation(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    messages = state["messages"]
    if len(messages) < MESSAGE_THRESHOLD:
        return state

    readable = []
    for m in messages:
        if isinstance(m, HumanMessage):
            readable.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                readable.append(f"Assistant: {content}")

    if not readable:
        return state

    summary_prompt = (
        "Summarise this travel assistant conversation in 3–5 sentences. "
        "Focus on destinations, user preferences, and decisions made.\n\n"
        + "\n".join(readable)
    )
    summary_msg = llm.invoke([
        SystemMessage(content="You are a concise conversation summariser."),
        HumanMessage(content=summary_prompt),
    ])
    logger.info(f"Summarised {len(messages)} messages.")

    summary_sys = SystemMessage(
        content=f"Summary of conversation so far:\n\n{summary_msg.content}\n\nContinue from here."
    )
    keep = messages[-2:] if len(messages) >= 2 else messages
    state["messages"] = [summary_sys] + keep
    return state


# ── routing ───────────────────────────────────────────────────────────────
def decide_next(state: RuntimeState) -> str:
    last = state["messages"][-1] if state["messages"] else None
    if state.get("tool_call_count", 0) >= MAX_TOOL_CALLS_PER_TURN:
        return "summarize"
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "execute_tools"
    return "summarize"


# ── compile graph ─────────────────────────────────────────────────────────
workflow = StateGraph(RuntimeState)
workflow.add_node("agent", respond_to_user)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("summarize", summarize_conversation)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    decide_next,
    {"execute_tools": "execute_tools", "summarize": "summarize"},
)
workflow.add_edge("execute_tools", "agent")
workflow.add_edge("summarize", END)

redis_saver = RedisSaver(redis_client=redis_client)
redis_saver.setup()
graph = workflow.compile(checkpointer=redis_saver)

print("✅ Graph compiled")

INFO langgraph.checkpoint.redis Redis client is a standalone client
INFO redisvl.index.index Index already exists, not overwriting.
INFO redisvl.index.index Index already exists, not overwriting.


✅ Graph compiled


# Cell 12 – Interactive loop (with selector and commands)

In [22]:
from langchain_core.messages import HumanMessage

def main(user_id: str = "demo_user", verbose: bool = False):

    # ── Conversation selector ────────────────────────────────────────────
    thread_id, _ = select_conversation(user_id=user_id)

    print(f"🌍 Travel Assistant with Memory (Snowflake Cortex + Redis)")
    print(f"   user={user_id}  thread={thread_id}")
    print("   Commands: exit | quit | debug | history | memories\n")

    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}, "recursion_limit": 50}
    state  = RuntimeState(messages=[], tool_call_count=0)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            save_transcript(state, thread_id, user_id)
            break

        if not user_input:
            continue

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            save_transcript(state, thread_id, user_id)
            break

        if user_input.lower() == "debug":
            verbose = not verbose
            print(f"  [verbose mode {'ON' if verbose else 'OFF'}]\n")
            continue

        if user_input.lower() == "history":
            print_transcript(thread_id, user_id)
            continue

        if user_input.lower() == "memories":
            keys = redis_client.keys("memory:*")
            print(f"\n🧠 Long-term memories ({len(keys)} total):")
            for key in sorted(keys):
                data = redis_client.json().get(key)
                if data:
                    ts  = data.get("created_at", "")[:19]
                    mt  = data.get("memory_type", "?")
                    cnt = data.get("content", "")
                    print(f"  [{mt:8s}] {ts}  {cnt}")
            print()
            continue

        # ── Reset tool‑call counter for new turn ─────────────────────────
        state["tool_call_count"] = 0
        state["messages"].append(HumanMessage(content=user_input))

        try:
            for result in graph.stream(state, config=config, stream_mode="values"):
                state = RuntimeState(**result)

            # Find the last clean assistant reply
            reply = None
            for m in reversed(state["messages"]):
                if isinstance(m, AIMessage):
                    content = m.content.strip() if m.content else ""
                    if content and not extract_tool_call(content):
                        reply = content
                        break

            print(f"\nAssistant: {reply if reply else '(no reply — try again)'}\n")
            save_transcript(state, thread_id, user_id)

            if verbose:
                print(f"── DEBUG (tool_call_count={state.get('tool_call_count', 0)}) ──")
                for i, m in enumerate(state["messages"]):
                    kind    = type(m).__name__
                    content = (m.content or "")[:120].replace("\n", " ")
                    tc      = f" [calls={[t['name'] for t in getattr(m, 'tool_calls', [])]}]" if getattr(m, "tool_calls", None) else ""
                    print(f"  [{i:02d}] {kind}{tc}: {content!r}")
                print("──────────────────────────────────────\n")

        except Exception as e:
            logger.error(f"Error during graph execution: {e}", exc_info=True)
            print(f"\n⚠️  Error: {e}\n")

    return state


if __name__ == "__main__":
    uid = input("Enter user ID (default demo_user): ") or "demo_user"
    final_state = main(user_id=uid, verbose=False)


  📚 CONVERSATION SELECTOR
  [0] 🆕 Start a new conversation

  [1] 🗂  thread_20260806_131441
       6 turns  |  "hello"

  [2] 🗂  thread_20260806_131846
       18 turns  |  "hi"

  [3] 🗂  thread_20260806_134036
       61 turns  |  "do the websearch for the rains on 7 august 2026 weather fore..."

  [4] 🗂  thread_20260806_153647
       13 turns  |  "whats my name"

  [5] 🗂  thread_20260806_154147
       14 turns  |  "name is rahmath hampi code is 583239 remember this. also ham..."

  [6] 🗂  thread_20260806_154345
       4 turns  |  "name?"

  [7] 🗂  thread_20260806_154501
       20 turns  |  "name is rahmath and hampi code is 548574 and tomorrows date?..."

  [8] 🗂  thread_20260806_154651
       4 turns  |  "name?"

  [9] 🗂  thread_20260806_202625
       8 turns  |  "weather of hampi"

  [10] 🗂  thread_20260807_051725
       21 turns  |  "whats my name?"

  [11] 🗂  thread_20260807_054921
       15 turns  |  "whats my name"

  [12] 🗂  thread_20260807_055429
       12 turns  |  "whats my 

ERROR __main__ Error during graph execution: Embed API error 400: {"code":"390400","message":"The request entity had the following errors: text size must be between 1 and 2147483647 (was [])","request_id":"e111dfc3-d4bd-40fb-a7c4-66699cf6d535","error_code":"390400"}
Traceback (most recent call last):
  File "C:\Users\rahma\AppData\Local\Temp\ipykernel_16000\1243774116.py", line 58, in main
    for result in graph.stream(state, config=config, stream_mode="values"):
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\pregel\main.py", line 2967, in stream
    for _ in runner.tick(
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\pregel\_runner.py", line 207, in tick
    run_with_retry(
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\pregel\_retry.py", line 617, in run_with_retry
    return task.proc.invoke(task.input, config)
  


⚠️  Error: Embed API error 400: {"code":"390400","message":"The request entity had the following errors: text size must be between 1 and 2147483647 (was [])","request_id":"e111dfc3-d4bd-40fb-a7c4-66699cf6d535","error_code":"390400"}


Assistant: Hampi is a village in northern Karnataka, India. It is located within the ruins of the city of Vijayanagara, the former capital of the Vijayanagara Empire.



INFO __main__ 💾 Transcript saved → conversation:demo_user:thread_20260807_071051 (3 turns)
WARNING __main__ Malformed LLM response — falling back to web search on user query.
INFO primp response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=whats%20news%20on%20hampi%20current 200
INFO primp response: https://grokipedia.com/api/typeahead?query=whats+news+on+hampi+current&limit=1 200
INFO primp response: https://yandex.com/search/site/?text=whats+news+on+hampi+current&web=1&searchid=7765482 200



Assistant: 🌐 Web search results for 'whats news on hampi current':

- **Hampi: Latest News and Updates, Top Stories... - Oneindia News**
  Hampi: Get Hampi latest news and headlines, top stories, live updates, speech highlights, special reports, articles, videos, photos and complete coverage at Oneindia.com.
  🔗 https://www.oneindia.com/topic/hampi

- **Hampi: Latest News, Photos, Videos on Hampi - NDTV.COM**
  Find Hampi Latest News, Videos & Pictures on Hampi and see latest updates, news, information from NDTV.COM. Explore more on Hampi.
  🔗 https://www.ndtv.com/topic/hampi

- **Hampi News: Latest Hampi News and Updates at News18**
  Read Politics news, current affairs and news headlines online on Hampi News today.
  🔗 https://www.news18.com/topics/hampi/

- **Hampi | Latest & Breaking News on Hampi - Moneycontrol.com**
  Read why Hampi should be on your 2025 travel bucket list! Explore ancient ruins, breathtaking landscapes, thrilling adventures, and hidden gems in this UNESCO Worl

...

INFO __main__ 💾 Transcript saved → conversation:demo_user:thread_20260807_071051 (7 turns)
INFO primp response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=todays%20date 200
INFO primp response: https://en.wikipedia.org/w/api.php?action=query&format=json&prop=extracts&titles=Today%27s%20Detective&explaintext=0&exintro=0&redirects=1 200
INFO primp response: https://grokipedia.com/api/typeahead?query=todays+date&limit=1 200
INFO primp response: https://www.mojeek.com/search?q=todays+date 200
INFO primp response: https://yandex.com/search/site/?text=todays+date&web=1&searchid=3469185 200
INFO __main__ Summarised 9 messages.



Assistant: 🌐 Web search results for 'todays date':

- **Today's Detective**
  The Ghost Detective (Korean: 오늘의 탐정) is a 2018 South Korean television series starring Choi Daniel, Park Eun-bin and Lee Ji-ah. It aired on KBS2 from September 5 to October 31, 2018, every Wednesday and Thursday at 22:00 (KST).
  🔗 https://en.wikipedia.org/wiki/Today's_Detective

- **What Is Today's Date? - Friday, July 10, 2026 | Current Date & Time**
  Get current date in multiple formats, programming examples, and date calculations. Updated in real-time.
  🔗 https://www.iamrohit.in/date-today/

- **timeanddate.com**
  Countdown to Any Date. Create your own countdown. Illustration of an egg timer and a countdwon.Weather forecast for the next hour, today, tomorrow, and 14 days ahead.
  🔗 https://www.timeanddate.com/

- **What Is Today’s Date?**
  Get to know what date it is today based on your current location. What Is the Date Today In Numbers? The current date depends on the region you are currently resid

INFO __main__ 💾 Transcript saved → conversation:demo_user:thread_20260807_071051 (10 turns)
ERROR __main__ Error during graph execution: Embed API error 400: {"code":"390400","message":"The request entity had the following errors: text size must be between 1 and 2147483647 (was [])","request_id":"06791752-88dc-4cfd-b2b8-86d99ed18dae","error_code":"390400"}
Traceback (most recent call last):
  File "C:\Users\rahma\AppData\Local\Temp\ipykernel_16000\1243774116.py", line 58, in main
    for result in graph.stream(state, config=config, stream_mode="values"):
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\pregel\main.py", line 2967, in stream
    for _ in runner.tick(
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\pregel\_runner.py", line 207, in tick
    run_with_retry(
  File "c:\Users\rahma\Downloads\Interview-prep\junaid-interview-prep\.venv\lib\site-packages\langgraph\prege


⚠️  Error: Embed API error 400: {"code":"390400","message":"The request entity had the following errors: text size must be between 1 and 2147483647 (was [])","request_id":"06791752-88dc-4cfd-b2b8-86d99ed18dae","error_code":"390400"}

Goodbye!


INFO __main__ 💾 Transcript saved → conversation:demo_user:thread_20260807_071051 (11 turns)


# Cell 13 – Inspect conversations (optional)

In [23]:
# List all transcripts for demo_user
keys = redis_client.keys("conversation:demo_user:*")
print("Transcripts:")
for k in sorted(keys):
    print(k.decode())

Transcripts:
conversation:demo_user:thread_20260806_131441
conversation:demo_user:thread_20260806_131846
conversation:demo_user:thread_20260806_134036
conversation:demo_user:thread_20260806_153647
conversation:demo_user:thread_20260806_154147
conversation:demo_user:thread_20260806_154345
conversation:demo_user:thread_20260806_154501
conversation:demo_user:thread_20260806_154651
conversation:demo_user:thread_20260806_202625
conversation:demo_user:thread_20260807_051725
conversation:demo_user:thread_20260807_054921
conversation:demo_user:thread_20260807_055429
conversation:demo_user:thread_20260807_060913
conversation:demo_user:thread_20260807_061316
conversation:demo_user:thread_20260807_064154
conversation:demo_user:thread_20260807_064437
conversation:demo_user:thread_20260807_070526
conversation:demo_user:thread_20260807_071051


In [24]:
# View a specific transcript — replace the thread id below
print_transcript("thread_20260807_054921", "demo_user")


TRANSCRIPT  user=demo_user  thread=thread_20260807_054921

[USER]
whats my name

[ASSISTANT]
I don't have any information about your name. This is the beginning of our conversation, and I don't have any prior knowledge about you. If you'd like to share your name, I can store it in my memory for our conversation.

[USER]
My name is Rahmath

[ASSISTANT]
✅ Stored your name as Rahmath.

[USER]
Hampi pin code is 583239

[ASSISTANT]
✅ Stored Hampi pin code as 583239.

[USER]
What is tomorrow's date?

[ASSISTANT]
Today is Friday, August 07, 2026. Tomorrow is Saturday, August 08, 2026.

[SUMMARY]
Summary of conversation so far:

Here is a summary of the conversation:

Rahmath, the user, shared their name and the pin code of Hampi (583239) with the travel assistant. No specific travel plans or destinations were discussed, but Hampi was mentioned. The conversation also established the current date as August 07, 2026, and the next day's date as August 08, 2026. No travel decisions or preferences

In [25]:
# List all long‑term memories (without embedding vectors)
keys = redis_client.keys("memory:*")
print(f"Total memories: {len(keys)}")
for key in sorted(keys):
    data = redis_client.json().get(key)
    if data:
        print(json.dumps({k: v for k, v in data.items() if k != "embedding"}, indent=2))
        print("-" * 40)

Total memories: 0
